# 🔬 Ví dụ: Sending tool results + Multi-turn với công cụ

> File bổ sung #3 — bám đúng 2 mục trong khóa học:
> - **Sending tool results** (trả kết quả công cụ về)
> - **Multi-turn conversations with tools** (trò chuyện nhiều lượt, Claude nhớ ngữ cảnh)

Mỗi phần có code chạy được + in `messages` để bạn THẤY chuyện gì xảy ra.


In [ ]:
import anthropic
client = anthropic.Anthropic()
MODEL = "claude-opus-4-8"

# 1 công cụ đơn giản: tra thời tiết (giả lập)
tools = [{
    "name": "get_weather",
    "description": "Lấy thời tiết hiện tại của một thành phố.",
    "input_schema": {"type": "object",
        "properties": {"city": {"type": "string"}}, "required": ["city"]},
}]

def get_weather(city):
    data = {"Hà Nội": "28°C, nắng nhẹ", "Đà Nẵng": "31°C, có mây", "TP.HCM": "33°C, oi"}
    return data.get(city, "Không có dữ liệu")

# PHẦN A — Sending tool results

## Trả 1 kết quả về cần đúng 3 quy tắc
1. Lưu lại lời Claude (`response.content`) — vai trò `assistant`
2. Kết quả nằm trong message `user`
3. `tool_use_id` khớp với `id` Claude tự sinh


In [ ]:
# Bước 1: khách hỏi -> Claude xin gọi tool
messages = [{"role": "user", "content": "Thời tiết Hà Nội thế nào?"}]
res = client.messages.create(model=MODEL, max_tokens=512, tools=tools, messages=messages)

print("stop_reason:", res.stop_reason)
tool_block = next(b for b in res.content if b.type == "tool_use")
print(f"Claude xin gọi: {tool_block.name}({tool_block.input}), id = {tool_block.id}")

In [ ]:
# Bước 2: (quy tắc 1) lưu lời Claude
messages.append({"role": "assistant", "content": res.content})

# Bước 3: chạy hàm thật
ket_qua = get_weather(tool_block.input["city"])
print("Hàm trả về:", ket_qua)

# Bước 4: (quy tắc 2 + 3) gửi kết quả về, role=user, khớp id
messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": tool_block.id,   # ← KHỚP id ở trên
        "content": ket_qua,
    }],
})

# Bước 5: gọi lại -> Claude trả lời
final = client.messages.create(model=MODEL, max_tokens=512, tools=tools, messages=messages)
print("\n💬 Claude:", final.content[0].text)

# PHẦN B — Multi-turn conversations with tools

## Điểm khác biệt: NHIỀU câu hỏi liên tiếp, Claude NHỚ ngữ cảnh

Bí quyết: **giữ một danh sách `messages` duy nhất** và cứ nối tiếp vào nó. Vì lịch sử được giữ, câu sau như *"thế còn X?"* Claude vẫn hiểu.

Ta viết 1 hàm `chat()` tái sử dụng được — nó tự lo vòng lặp gọi tool và **không xóa lịch sử** giữa các lượt.


In [ ]:
# Lịch sử DÙNG CHUNG cho cả cuộc trò chuyện (không reset!)
lich_su = []

def chat(cau_hoi_moi):
    lich_su.append({"role": "user", "content": cau_hoi_moi})

    while True:
        res = client.messages.create(model=MODEL, max_tokens=512,
                                     tools=tools, messages=lich_su)
        if res.stop_reason == "end_turn":
            tra_loi = next(b.text for b in res.content if b.type == "text")
            lich_su.append({"role": "assistant", "content": res.content})
            return tra_loi

        # Claude xin gọi tool
        lich_su.append({"role": "assistant", "content": res.content})
        results = []
        for b in res.content:
            if b.type == "tool_use":
                results.append({"type": "tool_result", "tool_use_id": b.id,
                                "content": get_weather(b.input["city"])})
        lich_su.append({"role": "user", "content": results})

In [ ]:
# LƯỢT 1
print("👤 Khách: Thời tiết Hà Nội?")
print("🤖", chat("Thời tiết Hà Nội thế nào?"))

# LƯỢT 2 — câu hỏi PHỤ THUỘC lượt trước ("thế còn...")
print("\n👤 Khách: Thế còn Đà Nẵng?")
print("🤖", chat("Thế còn Đà Nẵng?"))   # Claude HIỂU vẫn đang hỏi thời tiết

# LƯỢT 3 — câu hỏi dựa trên cả 2 lượt trước
print("\n👤 Khách: Trong 2 nơi đó, chỗ nào nóng hơn?")
print("🤖", chat("Trong 2 thành phố vừa hỏi, chỗ nào nóng hơn?"))

## Xem lịch sử đã phình to thế nào

Đây là "bộ nhớ" giúp Claude nhớ ngữ cảnh. Chú ý nó dài dần và trộn lẫn text + tool_use + tool_result.


In [ ]:
def in_lich_su(messages):
    for i, m in enumerate(messages):
        c = m["content"]
        if isinstance(c, str):
            print(f"[{i}] {m['role']}: {c}")
        else:
            for b in c:
                t = b["type"] if isinstance(b, dict) else b.type
                if t == "text":
                    txt = b["text"] if isinstance(b, dict) else b.text
                    print(f"[{i}] {m['role']} (text): {txt[:55]}")
                elif t == "tool_use":
                    name = b["name"] if isinstance(b, dict) else b.name
                    inp = b["input"] if isinstance(b, dict) else b.input
                    print(f"[{i}] {m['role']} (tool_use): {name}({inp})")
                elif t == "tool_result":
                    print(f"[{i}] {m['role']} (tool_result): {b['content']}")

print(f"Tổng {len(lich_su)} message trong lịch sử:\n")
in_lich_su(lich_su)

## Vì sao multi-turn HOẠT ĐỘNG?

```
LƯỢT 1: hỏi Hà Nội  → [user, assistant tool_use, user tool_result, assistant trả lời]
LƯỢT 2: "thế còn Đà Nẵng?"
        → vì lịch sử CÒN NGUYÊN lượt 1, Claude hiểu "thế còn" = vẫn hỏi thời tiết
LƯỢT 3: "chỗ nào nóng hơn?"
        → Claude đọc cả 2 kết quả (28°C vs 31°C) trong lịch sử → so sánh được
```

🔑 **Mấu chốt:** KHÔNG reset `lich_su` giữa các lượt. Nó tích lũy mọi thứ (cả tool_use/tool_result) → đó là cách Claude "nhớ".


---
## ✅ Tóm tắt

| Phần | Cốt lõi |
|---|---|
| **Sending tool results** | Trả 1 kết quả: append lời Claude (assistant) → append tool_result (user) với `tool_use_id` khớp |
| **Multi-turn with tools** | Giữ 1 `lich_su` duy nhất, nối tiếp qua các lượt → Claude nhớ ngữ cảnh, hiểu được "thế còn...", "chỗ nào hơn..." |

> Multi-turn = nhiều lần "Sending tool results" gộp lại trong cùng một lịch sử không bị xóa.
